# 04 · Đánh giá quyền riêng tư bằng tấn công (C2) và huấn luyện liên kết có DP

Bốn thí nghiệm, mỗi thí nghiệm trả lời một câu hỏi của phản biện:

| # | Câu hỏi | Tấn công / thí nghiệm | Kỳ vọng |
|---|---|---|---|
| A | Đặc trưng nhịp gõ gửi đi có làm lộ danh tính? | **Tái nhận dạng**: đoán người gõ từ đặc trưng 1 phút, trước/sau lớp bảo vệ (chuẩn hóa khởi động + lượng tử hóa + nhiễu Laplace ε) | độ chính xác giảm về gần 1/N; **đường cong privacy–utility** |
| B | Khung hình làm mờ mặt có dựng lại được mặt? | **Feature inversion**: huấn luyện decoder dựng mặt từ khung đã pixel hóa 6×8 (LFW) | SSIM thấp, nhận dạng ArcFace ~ ngẫu nhiên |
| C | Mô hình đã huấn luyện có lộ ai nằm trong dữ liệu? | **Membership inference** (ngưỡng mất mát), không DP vs DP-SGD | AUC tấn công ~0,5 khi bật DP |
| D | Huấn luyện không gom dữ liệu về một chỗ có kém đi nhiều? | Tập trung vs **FedAvg / FedProx / FedAvg + DP** (mô phỏng nhiều đơn vị) | mất mát độ chính xác nhỏ |

Cần chạy **notebook 01** trước (file `keystroke_windows.parquet` trên Drive). Thời gian: ~40–60 phút trên GPU T4.

Ghi chú cho bài báo: phần D mô phỏng nhiều client trên một máy (cùng thuật toán với Flower FedAvg/FedProx); nếu chỉ có một
server nội bộ thì ghi rõ là mô phỏng.

In [ ]:
!pip -q install xgboost insightface onnxruntime-gpu scikit-image opacus

### Mã dùng chung
Hai ô dưới ghi `feature_schema.json` (hợp đồng đặc trưng với trang web) và `eg_common.py` (công thức đặc trưng, baseline luật, mô hình hợp nhất). Không cần sửa.

In [ ]:
%%writefile feature_schema.json
{
 "version": 1,
 "window_seconds": 2,
 "modalities": [
  "face",
  "identity",
  "object",
  "audio",
  "browser",
  "keystroke",
  "answer",
  "system"
 ],
 "features": [
  {
   "name": "face_count_mean",
   "modality": "face",
   "scale": 1,
   "source": "client",
   "desc": "Số khuôn mặt trung bình"
  },
  {
   "name": "face_count_max",
   "modality": "face",
   "scale": 1,
   "source": "client",
   "desc": "Số khuôn mặt lớn nhất"
  },
  {
   "name": "face_missing_ratio",
   "modality": "face",
   "scale": 1,
   "source": "client",
   "desc": "Tỉ lệ mẫu không thấy mặt"
  },
  {
   "name": "yaw_mean",
   "modality": "face",
   "scale": 45,
   "source": "client",
   "desc": "Góc quay ngang trung bình (độ, ước lượng từ 6 điểm)"
  },
  {
   "name": "yaw_std",
   "modality": "face",
   "scale": 20,
   "source": "client",
   "desc": "Độ lệch chuẩn góc quay ngang"
  },
  {
   "name": "pitch_mean",
   "modality": "face",
   "scale": 30,
   "source": "client",
   "desc": "Góc ngẩng/cúi trung bình (độ)"
  },
  {
   "name": "pitch_std",
   "modality": "face",
   "scale": 15,
   "source": "client",
   "desc": "Độ lệch chuẩn góc ngẩng/cúi"
  },
  {
   "name": "head_away_ratio",
   "modality": "face",
   "scale": 1,
   "source": "client",
   "desc": "Tỉ lệ mẫu quay đầu quá ngưỡng (|yaw|>30° hoặc |pitch|>25°)"
  },
  {
   "name": "box_shift",
   "modality": "face",
   "scale": 0.2,
   "source": "client",
   "desc": "Dịch chuyển tâm hộp mặt (theo chiều rộng khung)"
  },
  {
   "name": "face_size",
   "modality": "face",
   "scale": 0.3,
   "source": "client",
   "desc": "Diện tích hộp mặt / diện tích khung"
  },
  {
   "name": "face_conf",
   "modality": "face",
   "scale": 1,
   "source": "client",
   "desc": "Độ tin cậy dò mặt trung bình"
  },
  {
   "name": "id_score",
   "modality": "identity",
   "scale": 1,
   "source": "server",
   "desc": "Điểm khớp gần nhất với ảnh đăng ký"
  },
  {
   "name": "id_checked",
   "modality": "identity",
   "scale": 1,
   "source": "server",
   "desc": "Có lần xác thực lại trong cửa sổ"
  },
  {
   "name": "id_mismatch",
   "modality": "identity",
   "scale": 1,
   "source": "server",
   "desc": "Lần xác thực gần nhất không khớp"
  },
  {
   "name": "obj_phone",
   "modality": "object",
   "scale": 1,
   "source": "server",
   "desc": "Xác suất điện thoại"
  },
  {
   "name": "obj_book",
   "modality": "object",
   "scale": 1,
   "source": "server",
   "desc": "Xác suất sách/giấy"
  },
  {
   "name": "obj_headphone",
   "modality": "object",
   "scale": 1,
   "source": "server",
   "desc": "Xác suất tai nghe"
  },
  {
   "name": "obj_person_extra",
   "modality": "object",
   "scale": 1,
   "source": "server",
   "desc": "Xác suất có người thứ hai (theo thân người)"
  },
  {
   "name": "obj_screen",
   "modality": "object",
   "scale": 1,
   "source": "server",
   "desc": "Xác suất màn hình/laptop phụ"
  },
  {
   "name": "obj_age",
   "modality": "object",
   "scale": 30,
   "source": "server",
   "desc": "Số giây kể từ khung gần nhất (độ tươi)"
  },
  {
   "name": "speech_ratio",
   "modality": "audio",
   "scale": 1,
   "source": "client",
   "desc": "Tỉ lệ thời gian có tiếng nói"
  },
  {
   "name": "speech_segments",
   "modality": "audio",
   "scale": 3,
   "source": "client",
   "desc": "Số đoạn nói bắt đầu trong cửa sổ"
  },
  {
   "name": "audio_energy",
   "modality": "audio",
   "scale": 1,
   "source": "client",
   "desc": "Năng lượng âm thanh (0..1, thang dB chuẩn hóa)"
  },
  {
   "name": "hidden_sec",
   "modality": "browser",
   "scale": 2,
   "source": "client",
   "desc": "Số giây trang bị ẩn (rời tab)"
  },
  {
   "name": "blur_sec",
   "modality": "browser",
   "scale": 2,
   "source": "client",
   "desc": "Số giây cửa sổ mất focus"
  },
  {
   "name": "tab_switches",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Số lần rời tab/cửa sổ"
  },
  {
   "name": "fullscreen_exits",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Số lần thoát toàn màn hình"
  },
  {
   "name": "not_fullscreen_sec",
   "modality": "browser",
   "scale": 2,
   "source": "client",
   "desc": "Số giây không ở chế độ toàn màn hình"
  },
  {
   "name": "window_small",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Cửa sổ bị thu nhỏ/chia đôi (0/1)"
  },
  {
   "name": "screen_extended",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Đang có màn hình mở rộng (0/1, chỉ Chromium)"
  },
  {
   "name": "screen_changes",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Số lần thay đổi cấu hình màn hình"
  },
  {
   "name": "paste_attempts",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Số lần cố dán"
  },
  {
   "name": "copy_attempts",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Số lần cố sao chép/cắt đề"
  },
  {
   "name": "drop_attempts",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Số lần cố kéo thả"
  },
  {
   "name": "contextmenu",
   "modality": "browser",
   "scale": 1,
   "source": "client",
   "desc": "Số lần mở menu chuột phải"
  },
  {
   "name": "chars_after_return",
   "modality": "browser",
   "scale": 20,
   "source": "client",
   "desc": "Số ký tự gõ trong 10 giây ngay sau khi quay lại tab"
  },
  {
   "name": "keys",
   "modality": "keystroke",
   "scale": 10,
   "source": "client",
   "desc": "Số phím (đã trừ phím do bộ gõ sinh)"
  },
  {
   "name": "dwell_rel",
   "modality": "keystroke",
   "scale": 1,
   "source": "client",
   "desc": "Thời gian giữ phím TB / mức khởi động"
  },
  {
   "name": "dwell_cv",
   "modality": "keystroke",
   "scale": 1,
   "source": "client",
   "desc": "Hệ số biến thiên thời gian giữ phím"
  },
  {
   "name": "flight_rel",
   "modality": "keystroke",
   "scale": 1,
   "source": "client",
   "desc": "Khoảng cách giữa hai phím TB / mức khởi động"
  },
  {
   "name": "flight_cv",
   "modality": "keystroke",
   "scale": 1,
   "source": "client",
   "desc": "Hệ số biến thiên khoảng cách giữa hai phím"
  },
  {
   "name": "burst_ratio",
   "modality": "keystroke",
   "scale": 1,
   "source": "client",
   "desc": "Tỉ lệ khoảng cách < 150 ms"
  },
  {
   "name": "pause_ratio",
   "modality": "keystroke",
   "scale": 1,
   "source": "client",
   "desc": "Tỉ lệ khoảng cách > 2 giây"
  },
  {
   "name": "backspace_ratio",
   "modality": "keystroke",
   "scale": 0.3,
   "source": "client",
   "desc": "Tỉ lệ phím xóa/sửa"
  },
  {
   "name": "compositions",
   "modality": "keystroke",
   "scale": 3,
   "source": "client",
   "desc": "Số lần bộ gõ IME ghép chữ"
  },
  {
   "name": "chars_per_sec",
   "modality": "keystroke",
   "scale": 5,
   "source": "client",
   "desc": "Ký tự/giây"
  },
  {
   "name": "question_switches",
   "modality": "answer",
   "scale": 1,
   "source": "client",
   "desc": "Số lần chuyển câu"
  },
  {
   "name": "answer_changes",
   "modality": "answer",
   "scale": 1,
   "source": "client",
   "desc": "Số lần đổi đáp án"
  },
  {
   "name": "time_on_question",
   "modality": "answer",
   "scale": 60,
   "source": "client",
   "desc": "Số giây đã ở câu hiện tại"
  },
  {
   "name": "progress",
   "modality": "answer",
   "scale": 1,
   "source": "client",
   "desc": "Tỉ lệ câu đã trả lời"
  },
  {
   "name": "time_vs_expected",
   "modality": "answer",
   "scale": 2,
   "source": "client",
   "desc": "(thời gian trả lời - kỳ vọng theo độ khó) / độ lệch, âm = nhanh bất thường"
  },
  {
   "name": "lag_ms",
   "modality": "system",
   "scale": 200,
   "source": "client",
   "desc": "Độ trễ khung hình giao diện làm bài (watchdog)"
  },
  {
   "name": "offline",
   "modality": "system",
   "scale": 1,
   "source": "client",
   "desc": "Mất kết nối trong cửa sổ (0/1, không coi là gian lận)"
  },
  {
   "name": "monitor_load",
   "modality": "system",
   "scale": 0.2,
   "source": "client",
   "desc": "Tỉ lệ thời gian CPU dành cho giám sát"
  }
 ],
 "model_input": "concat(clip(x/scale,-5,5) cho từng đặc trưng, mask theo phương thức 0/1) -> 54 + 8 = 62 chiều",
 "privacy": {
  "keystroke_laplace_epsilon": 2.0,
  "keystroke_quantize": 0.05,
  "warmup_seconds": 60
 }
}

In [ ]:
%%writefile eg_common.py
"""Mã dùng chung cho các notebook ExamGuard (được nhúng vào notebook bằng %%writefile để chạy độc lập trên Colab).

Các hàm đặc trưng ở đây là bản Python của frontend/src/monitor/features.ts — PHẢI giữ khớp công thức,
vì mô hình huấn luyện ở notebook sẽ chạy trên đặc trưng do trình duyệt thí sinh tính.
"""
from __future__ import annotations

import json
import math
from pathlib import Path

import numpy as np

# ----------------------------------------------------------------------------- schema
_HERE = Path(__file__).resolve().parent
SCHEMA_PATH = next(p for p in (_HERE / "feature_schema.json", _HERE.parent / "shared" / "feature_schema.json") if p.exists())
SCHEMA = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
FEATURES = [f["name"] for f in SCHEMA["features"]]
MODALITIES = SCHEMA["modalities"]
SCALES = np.array([float(f["scale"]) for f in SCHEMA["features"]], np.float32)
MOD_OF = {f["name"]: f["modality"] for f in SCHEMA["features"]}
MOD_SLICES = {}
for _m in MODALITIES:
    _idx = [i for i, f in enumerate(SCHEMA["features"]) if f["modality"] == _m]
    MOD_SLICES[_m] = slice(_idx[0], _idx[-1] + 1)
DIM = len(FEATURES) + len(MODALITIES)
WINDOW_S = SCHEMA["window_seconds"]


def model_input(values: dict, masks: dict) -> np.ndarray:
    """Giống backend FeatureSpec.model_input: clip(x/scale) * mask + mask."""
    x = np.array([float(values.get(n, 0.0) or 0.0) for n in FEATURES], np.float32)
    x = np.clip(x / SCALES, -5, 5)
    m = np.array([1.0 if masks.get(mod) else 0.0 for mod in MODALITIES], np.float32)
    for k, mod in enumerate(MODALITIES):
        x[MOD_SLICES[mod]] *= m[k]
    return np.concatenate([x, m]).astype(np.float32)


# ----------------------------------------------------------------------------- khuôn mặt
def head_pose(kps) -> tuple[float, float]:
    """kps: 6 điểm BlazeFace chuẩn hóa [(x,y)...]: mắt phải, mắt trái, mũi, miệng, tai phải, tai trái."""
    if kps is None or len(kps) < 4:
        return 0.0, 0.0
    re, le, nose, mouth = [np.asarray(k[:2], float) for k in kps[:4]]
    mid = (re + le) / 2
    eye = max(float(np.hypot(*(le - re))), 1e-6)
    roll = math.atan2(le[1] - re[1], le[0] - re[0])
    c, s = math.cos(-roll), math.sin(-roll)

    def rot(p):
        d = p - mid
        return np.array([d[0] * c - d[1] * s, d[0] * s + d[1] * c])

    n, m = rot(nose), rot(mouth)
    yaw = math.degrees(math.atan(n[0] / eye / 0.6))
    pitch = (n[1] / max(m[1], 1e-6) - 0.55) * 120
    return float(np.clip(yaw, -90, 90)), float(np.clip(pitch, -60, 60))


def face_window(samples: list[dict]) -> dict:
    """samples: [{count, yaw, pitch, cx, cy, size, conf}] trong một cửa sổ 2 giây."""
    n = len(samples) or 1
    pres = [s for s in samples if s["count"] > 0]
    yaws = [s["yaw"] for s in pres]
    pit = [s["pitch"] for s in pres]
    shift = sum(math.hypot(pres[i]["cx"] - pres[i - 1]["cx"], pres[i]["cy"] - pres[i - 1]["cy"]) for i in range(1, len(pres)))
    sd = lambda a: float(np.std(a, ddof=1)) if len(a) > 1 else 0.0  # noqa: E731
    return {
        "face_count_mean": float(np.mean([s["count"] for s in samples])) if samples else 0.0,
        "face_count_max": max([s["count"] for s in samples], default=0),
        "face_missing_ratio": (len(samples) - len(pres)) / n if samples else 0.0,
        "yaw_mean": float(np.mean(yaws)) if yaws else 0.0, "yaw_std": sd(yaws),
        "pitch_mean": float(np.mean(pit)) if pit else 0.0, "pitch_std": sd(pit),
        "head_away_ratio": (sum(1 for s in pres if abs(s["yaw"]) > 30 or abs(s["pitch"]) > 25) / len(pres)) if pres else 0.0,
        "box_shift": shift,
        "face_size": float(np.mean([s["size"] for s in pres])) if pres else 0.0,
        "face_conf": float(np.mean([s["conf"] for s in pres])) if pres else 0.0,
    }


# ----------------------------------------------------------------------------- nhịp gõ
def key_timings(events: list[tuple]) -> tuple[list, list, int, int]:
    """events: [(t_ms, 'down'|'up', code, ime_bool)] -> dwell, flight, số phím, số phím sửa."""
    down, dwell, flight = {}, [], []
    last, keys, edits = None, 0, 0
    for t, typ, code, ime in events:
        if ime:
            continue
        if typ == "down":
            if code in down:
                continue
            down[code] = t
            keys += 1
            if code in ("Backspace", "Delete"):
                edits += 1
            if last is not None:
                flight.append(t - last)
            last = t
        else:
            t0 = down.pop(code, None)
            if t0 is not None:
                dwell.append(t - t0)
    return [d for d in dwell if 0 < d < 2000], [f for f in flight if 0 < f < 30000], keys, edits


def keystroke_window(events, compositions: int, inserted_chars: int, seconds: float, base: dict | None) -> dict:
    dwell, flight, keys, edits = key_timings(events)
    fl = [f for f in flight if f < 5000]
    dm = float(np.mean(dwell)) if dwell else 0.0
    fm = float(np.mean(fl)) if fl else 0.0
    sd = lambda a: float(np.std(a, ddof=1)) if len(a) > 1 else 0.0  # noqa: E731
    return {
        "keys": keys,
        "dwell_rel": (dm / base["dwell"] if base and dm else (1.0 if dm else 0.0)),
        "dwell_cv": sd(dwell) / dm if dm else 0.0,
        "flight_rel": (fm / base["flight"] if base and fm else (1.0 if fm else 0.0)),
        "flight_cv": sd(fl) / fm if fm else 0.0,
        "burst_ratio": sum(f < 150 for f in flight) / len(flight) if flight else 0.0,
        "pause_ratio": sum(f > 2000 for f in flight) / len(flight) if flight else 0.0,
        "backspace_ratio": edits / keys if keys else 0.0,
        "compositions": compositions,
        "chars_per_sec": inserted_chars / max(seconds, 1e-3),
    }


PRIVATE_KEYSTROKE = ["dwell_rel", "dwell_cv", "flight_rel", "flight_cv", "burst_ratio", "pause_ratio"]


def privatize(v: dict, epsilon: float, step: float = 0.05, rng: np.random.Generator | None = None) -> dict:
    """Lượng tử hóa + nhiễu Laplace(b = step/ε) như trên máy thí sinh. ε = inf -> chỉ lượng tử hóa."""
    rng = rng or np.random.default_rng()
    out = dict(v)
    for k in PRIVATE_KEYSTROKE:
        if k in out:
            noise = 0.0 if not np.isfinite(epsilon) else rng.laplace(0, step / max(epsilon, 1e-6))
            out[k] = max(0.0, round((out[k] + noise) / step) * step)
    return out


_NON_CHAR = {"backspace", "delete", "shift", "shiftleft", "shiftright", "control", "controlleft", "controlright", "ctrl",
             "alt", "altleft", "altright", "meta", "metaleft", "metaright", "tab", "enter", "capslock", "escape", "esc",
             "arrowup", "arrowdown", "arrowleft", "arrowright", "home", "end", "pageup", "pagedown", "insert"}


def is_char_key(code) -> bool:
    c = str(code).lower()
    return c not in _NON_CHAR and not c.startswith(("f1", "f2", "f3", "f4", "f5", "f6", "f7", "f8", "f9"))


def session_keystroke_windows(events: list[tuple], warmup_s: float = 60, epsilon: float = float("inf"),
                              window_s: float = WINDOW_S, rng=None) -> list[dict]:
    """Cắt chuỗi phím (t_ms tuyệt đối) thành cửa sổ 2 giây với chuẩn hóa theo đoạn khởi động, như trình duyệt."""
    if not events:
        return []
    events = sorted(events, key=lambda e: e[0])
    t0 = events[0][0]
    base, warm_d, warm_f = None, [], []
    out, cur, k = [], [], 0
    end = events[-1][0]
    w_start = t0
    i = 0
    while w_start <= end:
        w_end = w_start + window_s * 1000
        cur = []
        while i < len(events) and events[i][0] < w_end:
            cur.append(events[i])
            i += 1
        if base is None:
            d, f, _, _ = key_timings(cur)
            warm_d += d
            warm_f += [x for x in f if x < 5000]
            if (w_end - t0) / 1000 >= warmup_s and len(warm_d) >= 30 and len(warm_f) >= 30:
                base = {"dwell": float(np.median(warm_d)), "flight": float(np.median(warm_f))}
        chars = sum(1 for e in cur if e[1] == "down" and not e[3] and is_char_key(e[2]))
        w = keystroke_window(cur, 0, chars, window_s, base)
        w = privatize(w, epsilon, rng=rng) if epsilon is not None else w
        d, f, _, _ = key_timings(cur)
        # thời gian tuyệt đối (ms): chỉ dùng để đánh giá rủi ro tái nhận dạng ở notebook 04 — trình duyệt KHÔNG gửi
        w["_dwell_ms"] = float(np.mean(d)) if d else 0.0
        w["_flight_ms"] = float(np.mean([x for x in f if x < 5000])) if f else 0.0
        w["_dwell_ms_std"] = float(np.std(d)) if len(d) > 1 else 0.0
        out.append(w)
        w_start = w_end
        k += 1
    return out


# ----------------------------------------------------------------------------- âm thanh
def audio_window(speech_flags: list[bool], rms: list[float]) -> dict:
    onsets = sum(1 for i, s in enumerate(speech_flags) if s and (i == 0 or not speech_flags[i - 1]))
    r = float(np.mean(rms)) if rms else 0.0
    return {"speech_ratio": float(np.mean(speech_flags)) if speech_flags else 0.0, "speech_segments": onsets,
            "audio_energy": float(np.clip((20 * math.log10(r) + 60) / 60, 0, 1)) if r > 0 else 0.0}


def time_vs_expected(seconds: float, expected: float) -> float:
    return float(np.clip((seconds - expected) / (0.5 * expected), -5, 5)) if expected else 0.0


# ----------------------------------------------------------------------------- baseline luật ngưỡng (khớp backend)
RULES = [
    ("MULTI_FACE", "face", 0.60, lambda v: v["face_count_max"] >= 2),
    ("NO_FACE", "face", 0.25, lambda v: v["face_missing_ratio"] >= 0.99),
    ("HEAD_AWAY", "face", 0.15, lambda v: v["head_away_ratio"] >= 0.5),
    ("ID_MISMATCH", "identity", 0.70, lambda v: v["id_mismatch"] >= 1),
    ("PHONE", "object", 0.60, lambda v: v["obj_phone"] >= 0.5),
    ("PERSON_EXTRA", "object", 0.50, lambda v: v["obj_person_extra"] >= 0.5),
    ("HEADPHONE", "object", 0.35, lambda v: v["obj_headphone"] >= 0.5),
    ("BOOK", "object", 0.30, lambda v: v["obj_book"] >= 0.5),
    ("SCREEN", "object", 0.30, lambda v: v["obj_screen"] >= 0.5),
    ("SPEECH", "audio", 0.20, lambda v: v["speech_ratio"] >= 0.3),
    ("TAB_AWAY", "browser", 0.40, lambda v: v["hidden_sec"] > 0 or v["tab_switches"] > 0),
    ("BLUR", "browser", 0.20, lambda v: v["blur_sec"] >= 1),
    ("FULLSCREEN_EXIT", "browser", 0.20, lambda v: v["fullscreen_exits"] > 0 or v["not_fullscreen_sec"] >= 2),
    ("WINDOW_SMALL", "browser", 0.15, lambda v: v["window_small"] >= 1),
    ("EXTENDED_SCREEN", "browser", 0.20, lambda v: v["screen_extended"] >= 1),
    ("PASTE", "browser", 0.40, lambda v: v["paste_attempts"] > 0 or v["drop_attempts"] > 0),
    ("COPY", "browser", 0.25, lambda v: v["copy_attempts"] > 0),
    ("TYPE_AFTER_RETURN", "browser", 0.40, lambda v: v["chars_after_return"] >= 20),
    ("TRANSCRIBE_RHYTHM", "keystroke", 0.20, lambda v: v["keys"] >= 10 and v["pause_ratio"] < 0.02
     and v["backspace_ratio"] < 0.02 and v["burst_ratio"] > 0.6 and v["flight_cv"] < 0.5),
    ("FAST_ANSWER", "answer", 0.15, lambda v: v["time_vs_expected"] <= -2.5),
]


def rule_score(values: dict, masks: dict) -> float:
    v = {n: float(values.get(n, 0.0)) for n in FEATURES}
    p = 1.0
    for _, mod, w, test in RULES:
        if masks.get(mod) and test(v):
            p *= 1 - w
    return 1 - p


def rule_score_matrix(X: np.ndarray) -> np.ndarray:
    """X: [N, DIM] (đầu vào mô hình) -> điểm luật. Giải chuẩn hóa bằng scale."""
    out = np.zeros(len(X), np.float32)
    for i, x in enumerate(X):
        vals = dict(zip(FEATURES, (x[:len(FEATURES)] * SCALES).tolist()))
        masks = {m: x[len(FEATURES) + k] > 0.5 for k, m in enumerate(MODALITIES)}
        out[i] = rule_score(vals, masks)
    return out


# ----------------------------------------------------------------------------- đánh giá
def recall_at_fpr(y: np.ndarray, s: np.ndarray, fpr: float) -> float:
    y, s = np.asarray(y).astype(bool), np.asarray(s, float)
    neg = np.sort(s[~y])[::-1]
    if not len(neg) or not y.any():
        return float("nan")
    k = int(np.floor(fpr * len(neg)))
    thr = neg[k] if k < len(neg) else -np.inf
    return float((s[y] > thr).mean())


def summarize(y, s) -> dict:
    from sklearn.metrics import average_precision_score, roc_auc_score

    y = np.asarray(y).astype(int)
    if y.min() == y.max():
        return {"n": len(y), "pos": int(y.sum())}
    return {"n": len(y), "pos": int(y.sum()), "auc": round(roc_auc_score(y, s), 4), "ap": round(average_precision_score(y, s), 4),
            "recall@fpr1%": round(recall_at_fpr(y, s, 0.01), 4), "recall@fpr5%": round(recall_at_fpr(y, s, 0.05), 4)}


# ----------------------------------------------------------------------------- mô hình hợp nhất (PyTorch)
def build_fusion_model(hidden: int = 64, d: int = 32, dropout: float = 0.1):
    import torch
    from torch import nn

    class FusionGRU(nn.Module):
        """Encoder riêng từng phương thức -> cổng (gated fusion, che phương thức thiếu) -> GRU nhân quả -> xác suất.

        Một bước = một cửa sổ 2 giây; trạng thái ẩn h mang thông tin quá khứ (streaming)."""

        def __init__(self):
            super().__init__()
            self.slices = [MOD_SLICES[m] for m in MODALITIES]
            self.enc = nn.ModuleList([nn.Sequential(nn.Linear(s.stop - s.start, d), nn.GELU(), nn.Linear(d, d))
                                      for s in self.slices])
            self.gate = nn.Linear(len(MODALITIES) * d, len(MODALITIES))
            self.drop = nn.Dropout(dropout)
            self.gru = nn.GRUCell(d, hidden)
            self.head = nn.Sequential(nn.Linear(hidden + d, hidden), nn.GELU(), nn.Linear(hidden, 1))
            self.hidden = hidden
            self.recon = nn.Linear(hidden + d, len(FEATURES))  # cho tiền huấn luyện tự giám sát

        def fuse(self, x):
            feats = x[..., :len(FEATURES)]
            mask = x[..., len(FEATURES):]
            z = torch.stack([enc(feats[..., s]) for enc, s in zip(self.enc, self.slices)], dim=-2)  # [..., M, d]
            logits = self.gate(z.flatten(-2)) + (mask - 1) * 1e4  # phương thức thiếu -> cổng = 0
            g = torch.softmax(logits, dim=-1)
            g = g * (mask.sum(-1, keepdim=True) > 0)
            fused = (g.unsqueeze(-1) * z).sum(-2)
            return self.drop(fused), g

        def step(self, x, h):
            fused, g = self.fuse(x)
            h2 = self.gru(fused, h)
            out = torch.cat([h2, fused], -1)
            return torch.sigmoid(self.head(out)).squeeze(-1), h2, g

        def forward(self, xs):  # xs: [B, T, D] -> logits [B, T], gates [B, T, M], recon [B, T, F]
            B, T, _ = xs.shape
            h = xs.new_zeros(B, self.hidden)
            logits, gates, recs = [], [], []
            for t in range(T):
                fused, g = self.fuse(xs[:, t])
                h = self.gru(fused, h)
                out = torch.cat([h, fused], -1)
                logits.append(self.head(out).squeeze(-1))
                gates.append(g)
                recs.append(self.recon(out))
            return torch.stack(logits, 1), torch.stack(gates, 1), torch.stack(recs, 1)

    return FusionGRU()


def export_fusion_onnx(model, path: str) -> None:
    """Xuất MỘT BƯỚC streaming: x[1,D], h[1,H] -> prob[1], h_out[1,H], gates[1,M] (đúng tên backend đọc)."""
    import torch

    class Step(torch.nn.Module):
        def __init__(self, m):
            super().__init__()
            self.m = m

        def forward(self, x, h):
            return self.m.step(x, h)

    model.eval()
    step = Step(model)
    x = torch.zeros(1, DIM)
    h = torch.zeros(1, model.hidden)
    torch.onnx.export(step, (x, h), path, input_names=["x", "h"], output_names=["prob", "h_out", "gates"],
                      opset_version=17, dynamo=False)
    import onnx

    m = onnx.load(path)
    m.ir_version = min(m.ir_version, 9)  # onnxruntime trên server có thể là bản cũ
    m.metadata_props.add(key="schema_version", value=str(SCHEMA["version"]))
    m.metadata_props.add(key="features", value=",".join(FEATURES))
    onnx.save(m, path)


def check_onnx_parity(model, path: str, xs: np.ndarray, atol: float = 1e-4) -> float:
    """Chạy chuỗi bằng ONNX từng bước, so với PyTorch chạy cả chuỗi."""
    import onnxruntime as ort
    import torch

    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    model.eval()
    with torch.no_grad():
        ref = torch.sigmoid(model(torch.from_numpy(xs[None]).float())[0])[0].numpy()
    h = np.zeros((1, model.hidden), np.float32)
    got = []
    for x in xs:
        p, h, _ = sess.run(None, {"x": x[None].astype(np.float32), "h": h})
        got.append(float(np.asarray(p).reshape(-1)[0]))
    err = float(np.max(np.abs(np.array(got) - ref)))
    assert err < atol, err
    return err


# ----------------------------------------------------------------------------- phiên thi tổng hợp
def ks_window(kind: str, rng, ks_bona=None, ks_copy=None) -> np.ndarray:
    pool = ks_bona if kind == "bona" else ks_copy
    if pool is not None and len(pool):
        return pool[rng.integers(len(pool))]
    # mô phỏng: [keys, dwell_rel, dwell_cv, flight_rel, flight_cv, burst, pause, backspace, compositions, cps]
    if kind == "bona":
        k = rng.poisson(8)
        return np.array([k, rng.normal(1, .12), rng.uniform(.3, .7), rng.normal(1, .25), rng.uniform(.6, 1.4),
                         rng.uniform(.1, .4), rng.uniform(0, .15), rng.uniform(.03, .15), rng.poisson(1), k / 2], np.float32)
    k = rng.poisson(14)
    return np.array([k, rng.normal(.95, .08), rng.uniform(.2, .4), rng.normal(.8, .1), rng.uniform(.3, .7),
                     rng.uniform(.4, .8), rng.uniform(0, .03), rng.uniform(0, .03), rng.poisson(1), k / 2], np.float32)


EPISODES = ["tab_lookup", "phone", "second_person", "whisper", "paste", "second_screen", "llm_transcribe"]


def synth_session(sid: int, T: int, cheat: bool, tier: str, rng, ks_bona=None, ks_copy=None) -> dict:
    """Một phiên thi tổng hợp: nền bình thường có nhiễu thực tế + 1–3 đoạn gian lận nếu cheat."""
    FI = {n: i for i, n in enumerate(FEATURES)}
    ks = lambda kind: ks_window(kind, rng, ks_bona, ks_copy)  # noqa: E731
    V = np.zeros((T, len(FEATURES)), np.float32)
    y = np.zeros(T, np.int64)
    f = lambda name: FI[name]  # noqa: E731
    # --- nền bình thường
    miss_p = {"low": .12, "mid": .04, "high": .02}[tier]
    present = rng.random(T) > miss_p
    V[:, f("face_count_mean")] = present
    V[:, f("face_count_max")] = present
    V[:, f("face_missing_ratio")] = ~present
    V[:, f("yaw_mean")] = rng.normal(0, 8, T) * present
    V[:, f("yaw_std")] = np.abs(rng.normal(3, 2, T)) * present
    V[:, f("pitch_mean")] = rng.normal(5, 6, T) * present
    V[:, f("pitch_std")] = np.abs(rng.normal(2, 1.5, T)) * present
    glance = rng.random(T) < .02
    V[glance, f("yaw_mean")] = rng.choice([-1, 1], glance.sum()) * rng.uniform(30, 50, glance.sum())
    V[:, f("head_away_ratio")] = (np.abs(V[:, f("yaw_mean")]) > 30).astype(np.float32)
    V[:, f("box_shift")] = np.abs(rng.normal(.01, .01, T))
    V[:, f("face_size")] = rng.uniform(.06, .14) * present
    V[:, f("face_conf")] = rng.uniform(.8, .97, T) * present
    V[:, f("id_score")] = rng.uniform(.55, .8)
    V[:, f("obj_age")] = rng.uniform(0, 10, T)
    cough = rng.random(T) < .01
    V[:, f("speech_ratio")] = cough * rng.uniform(.1, .3, T)
    V[:, f("speech_segments")] = cough
    V[:, f("audio_energy")] = rng.uniform(.05, .2, T) + cough * .3
    notif = rng.random(T) < .004  # thông báo hệ thống làm mất focus ngắn
    V[notif, f("blur_sec")] = rng.uniform(.3, 1.5, notif.sum())
    typing = rng.random(T) < .35
    for t in np.where(typing)[0]:
        V[t, MOD_SLICES["keystroke"]] = ks("bona")
    V[:, f("question_switches")] = rng.random(T) < .03
    V[:, f("answer_changes")] = rng.random(T) < .05
    V[:, f("time_on_question")] = rng.uniform(0, 120, T)
    V[:, f("progress")] = np.linspace(0, 1, T)
    V[:, f("time_vs_expected")] = np.where(rng.random(T) < .05, rng.normal(0, 1, T), 0)
    lag = {"low": 120, "mid": 40, "high": 15}[tier]
    V[:, f("lag_ms")] = np.abs(rng.normal(lag, lag / 2, T))
    off = rng.random(T) < .003
    V[:, f("offline")] = off
    V[:, f("monitor_load")] = {"low": .12, "mid": .05, "high": .02}[tier]
    # --- phương thức có/không theo phiên
    masks = {"face": 1, "identity": int(rng.random() < .8), "object": int(rng.random() < .7), "audio": int(rng.random() < .85),
             "browser": 1, "keystroke": 1, "answer": 1, "system": 1}
    # --- đoạn gian lận
    kinds = []
    if cheat:
        for _ in range(rng.integers(1, 4)):
            kind = EPISODES[rng.integers(len(EPISODES))]
            L = int(rng.integers(5, 45))
            s0 = int(rng.integers(10, max(11, T - L - 1)))
            sl = slice(s0, s0 + L)
            y[sl] = 1
            kinds.append(kind)
            if kind == "tab_lookup":
                V[sl, f("hidden_sec")] = 2.0
                V[sl, f("tab_switches")] = 0
                V[s0, f("tab_switches")] = 1
                V[sl, f("face_count_mean")] = 1
                after = slice(s0 + L, min(T, s0 + L + 5))
                V[after, f("chars_after_return")] = rng.uniform(20, 80)
                for t in range(after.start, after.stop):
                    V[t, MOD_SLICES["keystroke"]] = ks("copy")
                y[after] = 1
            elif kind == "phone":
                V[sl, f("pitch_mean")] = rng.uniform(25, 45)
                V[sl, f("head_away_ratio")] = 1
                V[sl, f("obj_phone")] = rng.uniform(.4, .9, L) * (rng.random(L) < .6)
            elif kind == "second_person":
                V[sl, f("face_count_max")] = (rng.random(L) < .5) + 1
                V[sl, f("obj_person_extra")] = rng.uniform(.3, .9, L) * (rng.random(L) < .6)
                V[sl, f("speech_ratio")] = rng.uniform(.2, .7, L)
                V[sl, f("speech_segments")] = rng.integers(1, 3, L)
            elif kind == "whisper":
                V[sl, f("speech_ratio")] = rng.uniform(.1, .5, L)
                V[sl, f("audio_energy")] = rng.uniform(.15, .35, L)
            elif kind == "paste":
                V[s0, f("paste_attempts")] = rng.integers(1, 3)
                y[sl] = 0
                y[s0:s0 + 3] = 1
            elif kind == "second_screen":
                V[sl, f("screen_extended")] = 1
                V[sl, f("yaw_mean")] = rng.choice([-1, 1]) * rng.uniform(30, 55, L)
                V[sl, f("head_away_ratio")] = 1
                V[sl, f("obj_screen")] = rng.uniform(.3, .8, L) * (rng.random(L) < .4)
            elif kind == "llm_transcribe":
                for t in range(sl.start, sl.stop):
                    V[t, MOD_SLICES["keystroke"]] = ks("copy")
                V[sl, f("time_vs_expected")] = rng.uniform(-4, -2)
    X = np.stack([model_input(dict(zip(FEATURES, row)), masks) for row in V])
    return {"id": f"syn:{sid}", "X": X, "y": y, "tier": tier, "source": "synthetic", "kinds": kinds}




def make_synthetic_sessions(n: int, T: int, rng, cheat_rate: float = 0.5, tiers=("low", "mid", "high"),
                            tier_p=(0.3, 0.5, 0.2), ks_bona=None, ks_copy=None) -> list[dict]:
    return [synth_session(i, T, cheat=rng.random() < cheat_rate, tier=rng.choice(list(tiers), p=list(tier_p)), rng=rng,
                          ks_bona=ks_bona, ks_copy=ks_copy) for i in range(n)]

In [ ]:
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

from eg_common import (DIM, FEATURES, MOD_OF, MOD_SLICES, MODALITIES, build_fusion_model, make_synthetic_sessions,
                       privatize, summarize)

OUT = Path("/content/examguard")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/examguard")
except Exception as e:
    print("Không mount được Drive:", e)
OUT.mkdir(parents=True, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
KFEATS = [f for f in FEATURES if MOD_OF[f] == "keystroke"]
kw = pd.read_parquet(OUT / "keystroke_windows.parquet")
print(kw.shape, kw.lang.value_counts().to_dict(), "người:", kw.user.nunique())

## A. Tái nhận dạng người gõ từ đặc trưng nhịp gõ
Kẻ tấn công (vd. người có quyền đọc CSDL) gom 30 cửa sổ liên tiếp (1 phút) của một người thành một mẫu và đoán người đó
trong N người. Huấn luyện trên câu hỏi 1–3, kiểm tra trên câu hỏi 4–6 (khác nội dung) của cùng người.

Ba cách biểu diễn:
1. **Thô**: thời gian giữ/chuyển phím tuyệt đối (ms) — thiết kế "ngây thơ" gửi nguyên nhịp gõ.
2. **Chuẩn hóa khởi động** (ε = ∞): chia cho trung vị 60 giây đầu của chính người đó, lượng tử hóa 0,05.
3. **+ nhiễu Laplace** với ε ∈ {8, 4, 2, 1, 0,5} (đúng lớp bảo vệ chạy trên trình duyệt).

Đồng thời đo **độ hữu ích**: AUC phát hiện câu trả lời có hỗ trợ LLM (XGBoost, chia theo người) ở từng mức ε.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from xgboost import XGBClassifier

RAW = ["_dwell_ms", "_flight_ms", "_dwell_ms_std", "keys", "burst_ratio", "pause_ratio", "backspace_ratio", "chars_per_sec"]
EPS = [float("inf"), 8, 4, 2, 1, 0.5]
POS = {"P", "T", "FP", "FT"}
users = kw.user.value_counts()
users = users[users >= 200].index  # đủ dữ liệu
base = kw[kw.user.isin(users) & kw.label.isin(["B", "P", "T"])].copy()
base["qnum"] = base.groupby("user")["sample"].rank(method="dense")
base["half"] = np.where(base.qnum <= base.groupby("user").qnum.transform("max") / 2, "train", "test")


def noisy(df: pd.DataFrame, eps: float, seed=0) -> pd.DataFrame:
    if not np.isfinite(eps):
        return df
    r = np.random.default_rng(seed)
    rows = [privatize(dict(zip(KFEATS, v)), eps, rng=r) for v in df[KFEATS].values]
    out = df.copy()
    out[KFEATS] = pd.DataFrame(rows, index=df.index)[KFEATS]
    return out


def minute_blocks(df: pd.DataFrame, cols: list[str], k=30) -> pd.DataFrame:
    rows = []
    for (u, half, sid), g in df.sort_values("t").groupby(["user", "half", "sample"]):
        for i in range(0, len(g) - k + 1, k):
            blk = g.iloc[i:i + k][cols]
            rows.append({"user": u, "half": half, **{f"{c}_m": v for c, v in blk.mean().items()},
                         **{f"{c}_s": v for c, v in blk.std().items()}})
    return pd.DataFrame(rows).fillna(0)


def reid_accuracy(df: pd.DataFrame, cols: list[str]) -> tuple[float, int]:
    B = minute_blocks(df, cols)
    tr, te = B[B.half == "train"], B[B.half == "test"]
    feats = [c for c in B.columns if c not in ("user", "half")]
    clf = RandomForestClassifier(500, random_state=SEED, n_jobs=-1).fit(tr[feats], tr.user)
    return float((clf.predict(te[feats]) == te.user).mean()), te.user.nunique()


def utility_auc(df: pd.DataFrame) -> float:
    g = df.groupby("sample")
    A = g[KFEATS].agg(["mean", "std", "median"]).fillna(0)
    A.columns = [f"{a}_{b}" for a, b in A.columns]
    meta = g[["user", "label"]].first()
    y = meta.label.isin(POS).astype(int).values
    us = meta.user.values
    scores = np.zeros(len(A))
    for tr, te in GroupKFold(5).split(A, groups=us):
        clf = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, eval_metric="logloss")
        clf.fit(A.iloc[tr], y[tr])
        scores[te] = clf.predict_proba(A.iloc[te])[:, 1]
    return summarize(y, scores)["auc"]


curve = []
acc, n = reid_accuracy(base, RAW)
curve.append({"biểu diễn": "Thô (ms tuyệt đối)", "eps": None, "reid_acc": acc, "chance": 1 / n, "utility_auc": None})
for eps in EPS:
    d = noisy(base, eps)
    acc, n = reid_accuracy(d, KFEATS)
    curve.append({"biểu diễn": "Chuẩn hóa" + ("" if not np.isfinite(eps) else f" + Laplace ε={eps}"), "eps": eps,
                  "reid_acc": acc, "chance": 1 / n, "utility_auc": utility_auc(d)})
curve = pd.DataFrame(curve)
curve.to_csv(OUT / "privacy_utility_keystroke.csv", index=False)
curve

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
c2 = curve.dropna(subset=["utility_auc"])
ax.plot(c2.reid_acc, c2.utility_auc, "o-")
for _, r in c2.iterrows():
    ax.annotate("ε=∞" if not np.isfinite(r.eps) else f"ε={r.eps:g}", (r.reid_acc, r.utility_auc), textcoords="offset points", xytext=(5, 5))
ax.axvline(curve.chance.iloc[0], ls="--", c="gray", label="đoán ngẫu nhiên")
ax.axvline(curve.reid_acc.iloc[0], ls=":", c="red", label="thô (ms)")
ax.set_xlabel("Độ chính xác tái nhận dạng (thấp = riêng tư hơn)")
ax.set_ylabel("AUC phát hiện hỗ trợ LLM (cao = hữu ích hơn)")
ax.legend()
ax.set_title("Đường cong privacy–utility (nhịp gõ)")
fig.tight_layout()
fig.savefig(OUT / "privacy_utility_keystroke.png", dpi=200)

## B. Dựng lại khuôn mặt từ khung đã làm mờ (feature inversion)
Trình duyệt pixel hóa vùng mặt (nới 1,5×1,7 lần) thành **6×8 khối** trước khi gửi. Ở đây làm khó cho phòng thủ hơn thực tế:
pixel hóa **chỉ đúng vùng mặt** của ảnh LFW (không nới), rồi huấn luyện mạng giải mã có giám sát (cặp mờ → gốc) trên các danh
tính huấn luyện; kiểm tra trên **danh tính chưa thấy**. Đo SSIM và nhận dạng 1:N bằng ArcFace (buffalo_l, chỉ dùng nghiên cứu).

In [ ]:
import cv2
from skimage.metrics import structural_similarity as ssim
from sklearn.datasets import fetch_lfw_people

lfw = fetch_lfw_people(min_faces_per_person=10, color=True, resize=1.0, slice_=(slice(60, 190), slice(60, 190)))
imgs = (lfw.images * (255 if lfw.images.max() <= 1 else 1)).astype(np.uint8)
imgs = np.stack([cv2.resize(im, (112, 112)) for im in imgs])  # RGB 112×112
ids = lfw.target
uid = np.unique(ids)
rng.shuffle(uid)
test_ids = set(uid[: len(uid) // 4])
is_test = np.array([i in test_ids for i in ids])
print(len(imgs), "ảnh,", len(uid), "người;", is_test.sum(), "ảnh kiểm tra của", len(test_ids), "người chưa thấy")


def pixelate(im: np.ndarray) -> np.ndarray:
    small = cv2.resize(im, (6, 8), interpolation=cv2.INTER_AREA)
    return cv2.resize(small, (112, 112), interpolation=cv2.INTER_NEAREST)


blur = np.stack([pixelate(im) for im in imgs])


class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Flatten(), nn.Linear(6 * 8 * 3, 512), nn.GELU(), nn.Linear(512, 256 * 7 * 7), nn.GELU())
        ups = []
        c = 256
        for co in (128, 64, 32, 16):
            ups += [nn.Upsample(scale_factor=2), nn.Conv2d(c, co, 3, padding=1), nn.BatchNorm2d(co), nn.GELU()]
            c = co
        self.dec = nn.Sequential(*ups, nn.Conv2d(c, 3, 3, padding=1), nn.Sigmoid())

    def forward(self, tiny):
        return self.dec(self.enc(tiny).view(-1, 256, 7, 7))


tiny = torch.tensor(np.stack([cv2.resize(im, (6, 8), interpolation=cv2.INTER_AREA) for im in imgs])).permute(0, 3, 1, 2).float() / 255
target = torch.tensor(imgs).permute(0, 3, 1, 2).float() / 255
dec = Decoder().to(DEV)
opt = torch.optim.Adam(dec.parameters(), 2e-3)
tr_idx = np.where(~is_test)[0]
for ep in range(40):
    for b in np.array_split(rng.permutation(tr_idx), max(1, len(tr_idx) // 64)):
        loss = nn.functional.l1_loss(dec(tiny[b].to(DEV)), target[b].to(DEV))
        opt.zero_grad()
        loss.backward()
        opt.step()
    if ep % 10 == 9:
        print("epoch", ep + 1, "L1", round(loss.item(), 4))
dec.eval()
with torch.no_grad():
    recon = torch.cat([dec(tiny[b].to(DEV)).cpu() for b in np.array_split(np.where(is_test)[0], 20)])
recon = (recon.permute(0, 2, 3, 1).numpy() * 255).astype(np.uint8)
te_idx = np.where(is_test)[0]

In [ ]:
from insightface.app import FaceAnalysis

fa = FaceAnalysis(name="buffalo_l", allowed_modules=["recognition"], providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
fa.prepare(ctx_id=0 if DEV == "cuda" else -1)
rec = fa.models["recognition"]


def emb(batch_rgb: np.ndarray) -> np.ndarray:
    out = []
    for i in range(0, len(batch_rgb), 128):
        f = rec.get_feat([im[..., ::-1] for im in batch_rgb[i:i + 128]])
        out.append(f / np.linalg.norm(f, axis=1, keepdims=True))
    return np.concatenate(out)


E_orig = emb(imgs[te_idx])
E_blur = emb(blur[te_idx])
E_rec = emb(recon)
test_lab = ids[te_idx]
# thư viện: ảnh gốc đầu tiên của mỗi người; mẫu thử: các ảnh còn lại
first = {}
for k, l in enumerate(test_lab):
    first.setdefault(l, k)
gal_k = np.array(list(first.values()))
gal_lab = test_lab[gal_k]
probe = np.array([k for k in range(len(te_idx)) if k not in set(gal_k)])


def top1(E):
    sims = E[probe] @ E_orig[gal_k].T
    return float((gal_lab[sims.argmax(1)] == test_lab[probe]).mean())


inv = pd.DataFrame([
    {"đầu vào nhận dạng": "Ảnh gốc (cận trên)", "top-1": top1(E_orig), "SSIM với gốc": 1.0},
    {"đầu vào nhận dạng": "Khung pixel hóa 6×8 (server nhận)", "top-1": top1(E_blur),
     "SSIM với gốc": float(np.mean([ssim(blur[i], imgs[i], channel_axis=2) for i in te_idx]))},
    {"đầu vào nhận dạng": "Mặt dựng lại bởi decoder", "top-1": top1(E_rec),
     "SSIM với gốc": float(np.mean([ssim(recon[k], imgs[i], channel_axis=2) for k, i in enumerate(te_idx)]))},
    {"đầu vào nhận dạng": "Đoán ngẫu nhiên", "top-1": 1 / len(gal_k), "SSIM với gốc": None},
])
inv.to_csv(OUT / "privacy_face_inversion.csv", index=False)
fig, axs = plt.subplots(3, 6, figsize=(9, 5))
for c in range(6):
    i = te_idx[c * 7]
    for r, im in enumerate([imgs[i], blur[i], recon[c * 7]]):
        axs[r, c].imshow(im)
        axs[r, c].axis("off")
for r, t in enumerate(["gốc", "pixel hóa", "dựng lại"]):
    axs[r, 0].set_title(t, loc="left", fontsize=9)
fig.tight_layout()
fig.savefig(OUT / "privacy_face_inversion.png", dpi=150)
inv

## C. Membership inference: không DP vs DP-SGD
Mô hình hợp nhất huấn luyện trên phiên **thành viên**; kẻ tấn công dùng mất mát của từng phiên để đoán phiên đó có nằm
trong dữ liệu huấn luyện không (AUC 0,5 = không đoán được). DP-SGD: cắt gradient **theo từng phiên** (đơn vị riêng tư =
một thí sinh) + nhiễu Gauss; ε tính bằng RDP accountant của Opacus.

In [ ]:
from opacus.accountants import RDPAccountant
from sklearn.metrics import roc_auc_score

KS_B = kw[kw.label == "B"][KFEATS].values.astype(np.float32)
KS_C = kw[kw.label.isin(list(POS))][KFEATS].values.astype(np.float32)
pool = make_synthetic_sessions(360, 150, rng, ks_bona=KS_B, ks_copy=KS_C)
members, nonmembers, test_s = pool[:120], pool[120:240], pool[240:]
Xm = torch.tensor(np.stack([s["X"] for s in members]))
Ym = torch.tensor(np.stack([s["y"] for s in members]))


def session_loss(model, sess) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        X = torch.tensor(np.stack([s["X"] for s in sess])).to(DEV)
        Y = torch.tensor(np.stack([s["y"] for s in sess])).float().to(DEV)
        logits, _, _ = model(X)
        return nn.functional.binary_cross_entropy_with_logits(logits, Y, reduction="none").mean(1).cpu().numpy()


def test_auc(model) -> float:
    model.eval()
    with torch.no_grad():
        X = torch.tensor(np.stack([s["X"] for s in test_s])).to(DEV)
        p = torch.sigmoid(model(X)[0]).cpu().numpy()
    return summarize(np.concatenate([s["y"] for s in test_s]), p.reshape(-1))["auc"]


def train_target(dp: bool, epochs=30, lr=2e-3, clip=1.0, sigma=1.0, batch=16):
    torch.manual_seed(SEED)
    model = build_fusion_model(dropout=0.0).to(DEV)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.Adam(params, lr)
    acct = RDPAccountant()
    q = batch / len(Xm)
    for ep in range(epochs):
        model.train()
        for idx in torch.randperm(len(Xm)).split(batch):
            if not dp:
                logits, _, _ = model(Xm[idx].to(DEV))
                loss = nn.functional.binary_cross_entropy_with_logits(logits, Ym[idx].float().to(DEV))
                opt.zero_grad()
                loss.backward()
                opt.step()
                continue
            # DP-SGD: gradient từng phiên -> cắt chuẩn -> cộng -> thêm nhiễu Gauss
            summed = [torch.zeros_like(p) for p in params]
            for i in idx:
                logits, _, _ = model(Xm[i:i + 1].to(DEV))
                loss = nn.functional.binary_cross_entropy_with_logits(logits, Ym[i:i + 1].float().to(DEV))
                g = torch.autograd.grad(loss, params)
                norm = torch.sqrt(sum((x ** 2).sum() for x in g))
                f = min(1.0, clip / (float(norm) + 1e-12))
                for s_, x in zip(summed, g):
                    s_.add_(x * f)
            for p, s_ in zip(params, summed):
                p.grad = (s_ + torch.randn_like(s_) * sigma * clip) / len(idx)
            opt.step()
            acct.step(noise_multiplier=sigma, sample_rate=q)
    eps = acct.get_epsilon(delta=1e-5) if dp else float("inf")
    return model, eps


mia = []
for dp, sigma in ((False, 0), (True, 0.8), (True, 1.5)):
    model, eps = train_target(dp, sigma=sigma)
    lm, ln = session_loss(model, members), session_loss(model, nonmembers)
    auc_attack = roc_auc_score(np.r_[np.ones(len(lm)), np.zeros(len(ln))], -np.r_[lm, ln])
    mia.append({"huấn luyện": "DP-SGD" if dp else "Thường", "sigma": sigma, "ε (δ=1e-5)": round(eps, 2),
                "AUC tấn công": round(auc_attack, 3), "AUC phát hiện (tập kiểm tra)": test_auc(model)})
mia = pd.DataFrame(mia)
mia.to_csv(OUT / "privacy_membership.csv", index=False)
mia

## D. Huấn luyện liên kết (federated): tập trung vs FedAvg / FedProx / FedAvg + DP
5 "đơn vị" (khoa/trường) với dữ liệu **không đồng nhất** (khác tỉ lệ máy yếu và tỉ lệ gian lận). Dữ liệu không rời đơn vị;
chỉ trọng số mô hình được gửi về và lấy trung bình. FedProx thêm phạt μ/2·‖w − w_global‖² để ổn định khi dữ liệu lệch.
DP-FedAvg: cắt chuẩn cập nhật mỗi đơn vị và thêm nhiễu Gauss vào trung bình.

In [ ]:
import copy

CLIENT_CFG = [((.7, .3, 0.), .3), ((.2, .6, .2), .5), ((0., .4, .6), .6), ((.5, .5, 0.), .2), ((.1, .3, .6), .7)]
clients = [make_synthetic_sessions(80, 150, rng, cheat_rate=cr, tier_p=tp, ks_bona=KS_B, ks_copy=KS_C) for tp, cr in CLIENT_CFG]
c_tensors = [(torch.tensor(np.stack([s["X"] for s in c])), torch.tensor(np.stack([s["y"] for s in c]))) for c in clients]


def local_train(model, X, Y, epochs=1, lr=2e-3, mu=0.0, global_params=None):
    opt = torch.optim.Adam(model.parameters(), lr)
    model.train()
    for _ in range(epochs):
        for idx in torch.randperm(len(X)).split(16):
            logits, _, _ = model(X[idx].to(DEV))
            loss = nn.functional.binary_cross_entropy_with_logits(logits, Y[idx].float().to(DEV))
            if mu > 0:
                loss = loss + mu / 2 * sum(((p - g) ** 2).sum() for p, g in zip(model.parameters(), global_params))
            opt.zero_grad()
            loss.backward()
            opt.step()
    return model


def federated(rounds=20, mu=0.0, dp_clip=None, dp_sigma=0.0):
    torch.manual_seed(SEED)
    glob = build_fusion_model().to(DEV)
    hist = []
    sizes = np.array([len(x) for x, _ in c_tensors], float)
    for r in range(rounds):
        gparams = [p.detach().clone() for p in glob.parameters()]
        updates = []
        for X, Y in c_tensors:
            local = copy.deepcopy(glob)
            local_train(local, X, Y, mu=mu, global_params=gparams)
            upd = [lp.detach() - gp for lp, gp in zip(local.parameters(), gparams)]
            if dp_clip:
                norm = torch.sqrt(sum((u ** 2).sum() for u in upd))
                upd = [u * min(1.0, dp_clip / (float(norm) + 1e-12)) for u in upd]
            updates.append(upd)
        w = sizes / sizes.sum()
        with torch.no_grad():
            for k, p in enumerate(glob.parameters()):
                avg = sum(w[c] * updates[c][k] for c in range(len(updates)))
                if dp_clip:
                    avg = avg + torch.randn_like(avg) * dp_sigma * dp_clip / len(updates)
                p.add_(avg)
        if r % 5 == 4:
            hist.append((r + 1, test_auc(glob)))
    return glob, hist


fed = []
central = build_fusion_model().to(DEV)
Xall = torch.cat([x for x, _ in c_tensors])
Yall = torch.cat([y for _, y in c_tensors])
local_train(central, Xall, Yall, epochs=20)
fed.append({"cách huấn luyện": "Tập trung (gom dữ liệu)", "AUC": test_auc(central)})
for name, kw_ in (("FedAvg", {}), ("FedProx μ=0.01", {"mu": 0.01}), ("FedAvg + DP (clip 1, σ 0.5)", {"dp_clip": 1.0, "dp_sigma": 0.5})):
    _, hist = federated(**kw_)
    fed.append({"cách huấn luyện": name, "AUC": hist[-1][1], "AUC theo vòng": hist})
fed = pd.DataFrame(fed)
fed.to_csv(OUT / "federated.csv", index=False)
fed

## Tổng hợp cho bài báo
Các file trên Drive: `privacy_utility_keystroke.{csv,png}`, `privacy_face_inversion.{csv,png}`, `privacy_membership.csv`,
`federated.csv`. Khi viết: nêu rõ thí nghiệm C, D dùng phiên tổng hợp; A dùng dữ liệu keystroke công khai; B dùng LFW.

In [ ]:
summary = {"keystroke_reid": curve.to_dict(orient="records"), "face_inversion": inv.to_dict(orient="records"),
           "membership": mia.to_dict(orient="records"), "federated": fed.drop(columns=["AUC theo vòng"], errors="ignore").to_dict(orient="records")}
json.dump(summary, open(OUT / "privacy_report.json", "w"), ensure_ascii=False, indent=1, default=str)
print(json.dumps(summary, ensure_ascii=False, indent=1, default=str)[:3000])